# Natural Language Processing (NLP) / Generative AI R&D
## HR CIO Resume Applicant Tracking System (AST)
### Portable Document Format (PDF) Reader 001 - Data Preparation

Read in a series of resumes and position descriptions.  Perform PII and prompt injection defense on the incoming input.  Save results to a versioned data file for later processing.


### Version History
+ v0.1 - Initial prototype with C Wood resume and various PDs.  Used combination of Doc2Vec, CountVectorizer, NLP and generative techniques.  Generative techniques proved most useful.
+ v0.2 - Added train set of PDs from HR CIO and resumes.  Target output anticipated based on hire.  Only Generative calculations, no NLP.  Employee class to hold data created.
+ v0.3 - Add prompt injection defense and PII defense.  Improved Prompt instructions. All output saved to datafile, only generative calculations, no NLP ML.  Updated JSON to proper format, add tensorrt to take advantage of GPU's.
+ v0.4 - Broke routines into 001, 002 for data preparation and saving of versioned output and then data processing (generative queries).
+ v0.6 - Batch of resumes (2nd) with updates applied from comment analysis work
+ v0.7 - Added PD evaluation to work history, soft skills, and hard skills generative factors
+ v0.8 - Added Gemini/GCP support, augmented returns and Prompt to adjust to Gemini inconsistent JSON returns.  Validation dataset executed and turned into customer.
+ v0.9 - Added specialized skills, incorporate .env (load_dotenv) variables.

### Defenses in order of execution:

Clean Named Entities

   + Searches for people's names and replaces it with "PERSON"
   
Clean PII

   + Searches for: email, credit card, hyperlinks, ips, street addresses and replaces them with same word.
   
Text Clean

   + Removes carriage returns, extra spaces, http references, @ symbols and various other characters not in the standard ASCII character set.
   
Prompt Injection

   + Uses a LLM to search the remaining text (now "cleansed") looking for suspicious input.
   + Keyword search for: ignore, pretend
   + Function "flags" for concern.
   

### TODO
+ Handle token excees and manage, summary of summaries.
+ Handle prompt engineering with library, use prompt templates.
+ Loop control for failed API calls due to time-out

### Privacy Information

+  https://github.com/ibm-ecosystem-engineering/Watson-NLP/blob/main/ML/PII-Extraction/PII%20Extraction%20-%20Pre-Trained%20Models.ipynb
+  https://medium.com/towards-generative-ai/personal-identifiable-information-pii-extraction-using-watson-nlp-library-c8e506a3dbe8
+  https://github.com/akazah/prompt-anonymizer
+  https://dzone.com/articles/an-easy-way-to-privacy-protect-a-dataset-using-pyt
+  https://medium.com/dropoutlabs/cape-python-apply-privacy-enhancing-techniques-to-protect-sensitive-data-in-pandas-and-spark-e0bf8c0d55db
+  https://www.tensorflow.org/responsible_ai/privacy/tutorials/classification_privacy
+  https://medium.com/sfu-cspmp/various-approaches-towards-data-privacy-with-python-differential-privacy-and-homomorphic-a748e560d43b
+  https://www.tensorflow.org/responsible_ai/privacy/tutorials/classification_privacy
+  https://github.com/leoli51/Names-Oracle
+  https://github.com/philipperemy/name-dataset
+  https://github.com/lmeulen/PrivacyFilter

### Prompt Injection Defense

+  https://haystack.deepset.ai/blog/how-to-prevent-prompt-injections#:~:text=By%20putting%20the%20user%20input%20into%20curly%20brackets%2C,a%20lower%20temperature%20and%20increasing%20the%20frequency%20penalty.
+  https://huggingface.co/collections/leonardlin/prompt-injection-65dd93985012ec503f2a735a
+  https://medium.com/google-cloud/generative-ai-protect-your-llm-against-prompt-injection-in-production-f99852910a8e
+  https://arxiv.org/pdf/2309.00614
+  https://genai.owasp.org/llmrisk/llm01-prompt-injection/
+  https://kai-greshake.de/posts/llm-malware/
+  https://www.researchsquare.com/article/rs-2873090/v1
+  https://arxiv.org/pdf/2306.05499
+  https://kai-greshake.de/posts/inject-my-pdf/
+  https://github.com/greshake/llm-security
+  https://aivillage.org/large%20language%20models/threat-modeling-llm/
+  https://research.kudelskisecurity.com/2023/05/25/reducing-the-impact-of-prompt-injection-attacks-through-design/
+  https://cobusgreyling.medium.com/the-introduction-of-chat-markup-language-chatml-is-important-for-a-number-of-reasons-5061f6fe2a85
+  https://hiddenlayer.com/research/prompt-injection-attacks-on-llms/
+  https://blog.seclify.com/prompt-injection-cheat-sheet/

### References:

+ https://github.com/alcidesmorales/LLMS-Resume-Matcher
+ https://medium.com/@kirudang/job-resume-matching-part-1-2-obtaining-similarity-score-using-doc2vec-a6d07fe3b355
+ https://medium.com/thedeephub/resume-scanner-leverage-the-power-of-llm-to-improve-your-resume-401a0cb49cd7
+ https://pantherax.com/how-to-use-llms-for-text-matching-and-similarity/
+ https://learn.microsoft.com/en-us/azure/ai-services/openai/tutorials/embeddings?tabs=python-new%2Ccommand-line&pivots=programming-language-python


### Technical References

+ https://github.com/srbhr/Resume-Matcher/tree/main
+ https://www.analyticsvidhya.com/blog/2021/06/resume-screening-with-natural-language-processing-in-python/
+ https://oindrilasen.com/2021/05/build-resume-scanner-using-python-nlp/
+ https://towardsdatascience.com/resume-screening-with-python-1dea360be49b


### Jupyte Notebook Hints
https://jupyter-tutorial.readthedocs.io/en/24.1.0/notebook/shortcuts.html

In [2]:
# -*- coding: utf-8 -*-

### Environment Validation

Using GCP or Azure read in arrays representing minimal library requirements (which might not be present in a Google Colab environment) and install / load the libraries as required.  Additional imports for standard libraries and tailored content to follow.

In [3]:
###########################################
#- Minimal imports to start
###########################################
try:
    import sys
    import subprocess
    import importlib.util
    import atexit
except ImportError as e:
    print("There was a problem importing the most basic libraries necessary for this code.")
    print(repr(e))
    raise SystemExit("Stop right there!")

###########################################
#- Final Exit Routine
###########################################
@atexit.register
def goodbye():
    print("GOODBYE")

###########################################
#- Cloud Environment Setup (Priming)
###########################################
# variables establishing environments
ENV_GCP=0
ENV_AZURE=1
user_input=-1
environments=["GCP", "Azure"]
    
#prompt user for environment before continuing
user_input = 1
while True:
  try:
     if user_input > -1:
         break;
     user_input = int(input("Select the environment you're running: (0) GCP (1) Azure"))     
     if user_input > 1:
         print("Not a valid choice, please try again.")
         continue;
  except ValueError:
     print("Not a valid choice, please try again.")
     continue
  else:
     print(f"Environment selected is: {environments[user_input]}")
     break 
        
############################################
#- Import a custom library, in this case a fairly useful logging framework
############################################
from pathlib import Path
debug_lib_location = Path("../ML-Support")
sys.path.append(str(debug_lib_location))
try:
  import debug
  debug.msg_debug("...debug library loaded.")
except ImportError as e:
  print("There was a problem importing the debug library.")
  print(repr(e))
  raise SystemExit("Without the debug library this code will note run.")


libraries=["transformers", "langchain", "openpyxl", "backoff", "spacy", "spacy-transformers", "numba","unidecode", "nltk",
           "alive-progress", "tqdm", "pyspellchecker", "wordcloud", "langchain", "icecream", "streamlit", 
           "fitz","dataclasses", "commonregex", "transformers", "spacy", "PyMuPDF", "PyPDF2", "pdfminer", 
           "pdfplumber","pdf2image","pytesseract", "pillow"]    
debug.msg_info(f"Validating environment for the following pip packages: {libraries}")

#load environment for non-generative libraries
try:
    for library in libraries:
      if library == "Pillow":
        spec = importlib.util.find_spec("PIL")
      else:
        spec = importlib.util.find_spec(library)
      if spec is None:
        print("...installing library " + library)
        subprocess.run(["pip", "install" , library, "--quiet"])
      else:
        print("...library " + library + " already installed.")          
except (subprocess.CalledProcessError, Exception) as e:
    print("Error: Failed to install required packages, your code might not run properly.")
    print(repr(e))


#load environment specific libraries for generative AI.
try:    
    if environments[user_input]=="GCP":
      subprocess.run(["pip", "install" , "--upgrade", "google-cloud-aiplatform", "--quiet"])
      gcp_libraries=["google-generativeai", "google-cloud-secret-manager"]
      for library in gcp_libraries:
        spec = importlib.util.find_spec(library)
        if spec is None:
          print("...installing library " + library)
          try:
              subprocess.run(["pip", "install" , library, "--quiet"])
          except (subprocess.CalledProcessError, Exception) as e:
              print("Error: Failed to install required packages, your code might not run properly.")
              print(repr(e))
        else:
          print("...library " + library + " already installed.")
    
        from google.cloud import aiplatform
        import vertexai.preview
        from google.cloud import secretmanager
    elif environments[user_input]=="Azure":
      azure_libraries=["openai", ]
      for library in azure_libraries:
        spec = importlib.util.find_spec(library)
        if spec is None:
          print("...installing library " + library)
          try:
              subprocess.run(["pip", "install" , library, "--quiet"])
          except (subprocess.CalledProcessError, Exception) as e:
              print("Error: Failed to install required packages, your code might not run properly.")
              print(repr(e))
        else:
          print("...library " + library + " already installed.")
    else:
        print("There was a problem processing your request.  Only numeric input of 0 or 1 is allowed.")
        print("Continued operations is not possible without the proper installed tools.")
        raise SystemExit("Stop right there!")
except Exception as e:
    print("There was a problem processing library installs for Generative AI libraries")
    print(repr(e))
    raise SystemExit("Stop right there!")

debug.msg_debug("...dynamic environment installs complete.")

[2024-12-05 02:23:13 UTC]   DEBUG: ...debug library loaded. 
[2024-12-05 02:23:13 UTC]    INFO: Validating environment for the following pip packages: ['transformers', 'langchain', 'openpyxl', 'backoff', 'spacy', 'spacy-transformers', 'numba', 'unidecode', 'nltk', 'alive-progress', 'tqdm', 'pyspellchecker', 'wordcloud', 'langchain', 'icecream', 'streamlit', 'fitz', 'dataclasses', 'commonregex', 'transformers', 'spacy', 'PyMuPDF', 'PyPDF2', 'pdfminer', 'pdfplumber', 'pdf2image', 'pytesseract', 'pillow'] 
...library transformers already installed.
...library langchain already installed.
...library openpyxl already installed.
...library backoff already installed.
...library spacy already installed.
...installing library spacy-transformers
...library numba already installed.
...library unidecode already installed.
...library nltk already installed.
...installing library alive-progress
...library tqdm already installed.
...installing library pyspellchecker
...library wordcloud already insta

### Additional Libraries

Read in core libraries, local logging framework, data science tools, Natural Language Processing Toolkit (NLTK), Spacy and other tools to support the effort.

In [4]:
debug.msg_info("Library imports")    
############################################
# INCLUDES
############################################

# +++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
# a set of libraries that perhaps should always be in Python source
# +++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
debug.msg_debug("...core libraries.")
import os
import datetime
import gc
import socket
import sys
import getopt
import inspect
import traceback
import warnings
import json
import pickle
from pathlib import Path
import itertools
import datetime
import re
import shutil
import string
from io import StringIO
import tqdm


import io
import math
import textwrap
import random
import glob
import time
from time import perf_counter
import subprocess
from multiprocessing import Pool
import backoff

# +++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
# Function Profiling
# +++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
import cProfile
import pstats
import io
from pstats import SortKey


# +++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
# Data Science Libraries
# +++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
debug.msg_debug("...classic data science libraries.")

#optimization routines
#from numba import jit
import numpy as np
import scipy as sp
#from sklearn.linear_model import LinearRegression


# +++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
# Additional libraries for this work
# +++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
debug.msg_debug("...application specific libraries.")
import math
from base64 import b64decode
from IPython.display import Image
import requests
from bs4 import BeautifulSoup                 #used to parse the text
from wordcloud import WordCloud, STOPWORDS    #custom library specifically designed to make word clouds
from spellchecker import SpellChecker
import fitz
#to handle strange characters
from unidecode import unidecode 


# +++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
# Persistence and Streaming Apps
# +++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
import streamlit as st

# +++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
# Graphics
# +++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
debug.msg_debug("...graphics.")
#import PIL
from PIL import Image
import PIL.ImageOps
#import matplotlib as matplt
#import matplotlib.pyplot as plt
import matplotlib as plt

# +++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
# progress bar
# +++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
debug.msg_debug("...progress bars.")
from alive_progress import alive_bar
#from alive_progress.styles import showtime, Show
from tqdm.notebook import trange, tqdm
#from tqdm import trange, tqdm

# +++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
#- PII libraries (regular expressions)
# +++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
debug.msg_debug("...regular expressions for PII and transformers for prompt injection defense.")
from commonregex import CommonRegex
from commonregex import email
from commonregex import time
from commonregex import credit_card
from commonregex import ip
from commonregex import ipv6
from commonregex import link
from commonregex import phone
from commonregex import street_address
from commonregex import btc_address

debug.msg_debug("...spacy (pii defense).")
import spacy
from spacy.language import Language
from spacy.tokens import Doc
from spacy.matcher import Matcher

debug.msg_debug("...hugging face model support.")
#injection defense
from transformers import pipeline

# +++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
#- Tensorflow AI/ML libraries (seek to use GPU's)
# +++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
debug.msg_debug("...Tensorflow")
#load first
try:
    import tensorrt
    assert tensorrt.Builder(tensorrt.Logger())
except ImportError as ie:
    debug.msg_warning("Failed to import TensorRT, not critical but worthy of noting")
    pass

try:
    #load second
    import tensorflow as tf
except ImportError as ie:
    debug.msg_warning("Failed to import TensorFlow, this could cause your code to fail.  Be forwarned.")
    pass
    
try:    
    import cudf
except ImportError as ie:
    debug.msg_warning("Failed to import CUDF, you will be using standard Pandas, not that you might not have GPU support.")
    pass
    
try:    
    import torch
except ImportError as ie:
    debug.msg_warning("Failed to Pytorch, this could seriously impact SpaCy and other functionality, be forewarned.")
    debug.msg_warning(f"...{repr(ie)}")
    pass
except Exception as e:
    debug.msg_warning("Failed to Pytorch, this could seriously impact SpaCy and other functionality, be forewarned.")
    debug.msg_warning(f"...{repr(e)}")
    pass

import pandas as pd
# +++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
#- NLTK required resources
# +++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
debug.msg_debug("...natural language processing.")
import nltk
from nltk.stem import PorterStemmer  # A word stemmer based on the Porter stemming algorithm.  Porter, M. "An algorithm for suffix stripping." Program 14.3 (1980): 130-137.
from nltk.stem import WordNetLemmatizer
from nltk import pos_tag
from nltk.tree import tree
#from nltk.book import *
from nltk import FreqDist
from nltk import sent_tokenize, word_tokenize
from nltk.corpus import stopwords    

nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download("words")
nltk.download("stopwords")
#nltk.download('averaged_perceptron_tagger')      #looks like you have to download select neural layers for specific functions, head to read the erorr output to learn this.


[2024-12-05 02:23:22 UTC]    INFO: Library imports 
[2024-12-05 02:23:22 UTC]   DEBUG: ...core libraries. 
[2024-12-05 02:23:22 UTC]   DEBUG: ...classic data science libraries. 
[2024-12-05 02:23:22 UTC]   DEBUG: ...application specific libraries. 
[2024-12-05 02:23:23 UTC]   DEBUG: ...graphics. 
[2024-12-05 02:23:23 UTC]   DEBUG: ...progress bars. 
[2024-12-05 02:23:23 UTC]   DEBUG: ...regular expressions for PII and transformers for prompt injection defense. 
[2024-12-05 02:23:23 UTC]   DEBUG: ...spacy (pii defense). 
[2024-12-05 02:23:27 UTC]   DEBUG: ...hugging face model support. 


/opt/conda/lib/python3.10/site-packages/transformers/utils/generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
2024-12-05 02:23:27.939945: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:485] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2024-12-05 02:23:27.960584: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:8454] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2024-12-05 02:23:27.966819: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1452] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2024-12-05 02:23:27.982426: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimiz

[2024-12-05 02:23:32 UTC]   DEBUG: ...Tensorflow 


/opt/conda/lib/python3.10/site-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(


[2024-12-05 02:23:34 UTC] WARNING: Failed to import CUDF, you will be using standard Pandas, not that you might not have GPU support. 
[2024-12-05 02:23:34 UTC]   DEBUG: ...natural language processing. 


[nltk_data] Downloading package punkt to /home/jupyter/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /home/jupyter/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package words to /home/jupyter/nltk_data...
[nltk_data]   Package words is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     /home/jupyter/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

## Functions

In [5]:
def set_library_configuration() -> None:
    
    ############################################
    #- JUPYTER NOTEBOOK OUTPUT CONTROL / FORMATTING
    ############################################
    #pandas set floating point to 4 places to things don't run loose
    debug.msg_info("Setting Pandas and Numpy library options.")    
    pd.set_option('display.max_colwidth', 10) # None if you want to view the full json blob in the printed dataframe, use this
    pd.options.display.float_format = '{:,.4f}'.format
    np.set_printoptions(precision=4)

In [6]:
def profile_function(func):
    def wrapper(*args, **kwargs):
        pr = cProfile.Profile()
        pr.enable()
        result = func(*args, **kwargs)
        pr.disable()
        s = io.StringIO()
        sortby = SortKey.CUMULATIVE
        ps = pstats.Stats(pr, stream=s).sort_stats(sortby)
        ps.print_stats()
        print(s.getvalue())
        return result
    return wrapper

### Define Dataclass

A Dataclass to support holding the data for each resume with information like name, original text, summary, spelling, and the other factors for evaluation.

In [7]:
from dataclasses import dataclass, field

## Dataclass used to represent each prospective employee and hold complex data
#  until the end of execution for save to a datafile and Power BI presentation.
#
@dataclass
class Employee:
    #filename, candidate
    name: str

    #original unaltered data
    original: str

    #PII scrubbed and confirmed to be valid from prompt injection defense
    cleansed: str


    #generative AI created from cleansed
    summary: str = field(init=False, default="Unknown")

    #analysis performed on "original" data
    spelling: float = field(init=False, default=0.0)
    
    #hashmap of pd's along with the JSON response of each PD.
    pd_json_response: [] = field(init=False,default_factory=list)
    
    #prompt injection, did one occur, if so prompt_injection should be populated
    prompt_injection_detected: bool = field(init=False,default=False)

    #prompt injection concerns, arrays of strings that "pop" from the analysis
    prompt_injection: [] = field(init=False,default_factory=list)

## Function Declaration

#### Custom Exception Display

In [8]:
## Manages exception output.
#  @param   (Exception)             - Exception to expound upon
#  @returns (None)                  - None
def process_exception(inc_exception) -> None:
    print(f"{BOLD_START}(Exception encountered):{BOLD_END} {type(inc_exception).__name__}")
    print(f"Details: {str(inc_exception)}")
    print("Traceback:")
    traceback.print_exc()

#### Library Manifest Display

In [9]:
## Outputs library version history of effort.
#
#  @returns (None)                  - None
def lib_diagnostics() -> None:

    import pkg_resources
    
    debug.msg_info(f"Entering {__name__} {inspect.stack()[0][3]}") 
    
    package_name_length=40
    package_version_length=20

    # Get installed packages
    the_packages=["cupy", "jupyter-core", "langchain", "langchain-core", "nltk", "numba", "numpy", "pandas", "pydantic", "pyspellchecker", "spacy", "scipy", "scikit-learn", "seaborn", "usaddress", "xarray",]
    the_packages.sort()
    
    installed_dict = {pkg.key: pkg.version for pkg in pkg_resources.working_set}
    installed=list(installed_dict.keys())
    installed.sort()
    
    #for package_idx, package_name in enumerate(installed):
    for idx, name in enumerate(installed):
         if name in the_packages:
             installed_version = installed_dict[name]
             print(f"{name:<40}#: {str(pkg_resources.parse_version(installed_version)):<20}")
   
    try:
        print(f"{'TensorFlow version':<40}#: {str(tf.__version__):<20}")
        print(f"{'     gpu.count:':<40}#: {str(len(tf.config.experimental.list_physical_devices('GPU')))}")
        print(f"{'     cpu.count:':<40}#: {str(len(tf.config.experimental.list_physical_devices('CPU')))}")
    except Exception as e:
        pass

    try:
        print(f"{'Torch version':<40}#: {str(torch.__version__):<20}")
        print(f"{'     GPUs available?':<40}#: {torch.cuda.is_available()}")
        print(f"{'     count':<40}#: {torch.cuda.device_count()}")
        print(f"{'     current':<40}#: {torch.cuda.current_device()}")
    except Exception as e:
        pass


    try:
      print(f"{'OpenAI Azure Version':<40}#: {str(the_openai_version):<20}")
    except Exception as e:
      pass

    print(f"{BOLD_START}List Devices{BOLD_END} #########################################")
    try:
      from tensorflow.python.client import device_lib
      print(device_lib.list_local_devices())
      print("")
    except RuntimeError as e:
      # Visible devices must be set before GPUs have been initialized
      print(str(repr(åe)))

    print(f"{BOLD_START}Devices Counts{BOLD_END} ########################################")
    try:
      print(f"Num GPUs Available: {str(len(tf.config.experimental.list_physical_devices('GPU')))}" )
      print(f"Num CPUs Available: {str(len(tf.config.experimental.list_physical_devices('CPU')))}" )
      print("")
    except RuntimeError as e:
      # Visible devices must be set before GPUs have been initialized
      print(str(repr(e)))

    print(f"{BOLD_START}Optional Enablement{BOLD_END} ####################################")
    try:
      gpus = tf.config.experimental.list_physical_devices('GPU')
    except RuntimeError as e:
      # Visible devices must be set before GPUs have been initialized
      print(str(repr(e)))

    if gpus:
      # Restrict TensorFlow to only use the first GPU
      try:
        tf.config.experimental.set_visible_devices(gpus[0], 'GPU')
        logical_gpus = tf.config.experimental.list_logical_devices('GPU')
        print( str( str(len(gpus)) + " Physical GPUs," + str(len(logical_gpus)) + " Logical GPU") )
      except RuntimeError as e:
        # Visible devices must be set before GPUs have been initialized
        print(str(repr(e)))
      print("")
        
    debug.msg_info(f"Exiting {__name__} {inspect.stack()[0][3]}") 
    return

#### String Manipulation (Data String Cleanup Routines)

In [10]:
  
## Read the contents of a text input and remove URL's, extra spaces, carriage returns, etc..
#
#  @param (employee)            - Employee Class (to pull data)
#  @returns (employee, float)   - Altered employee record, duration of execution
#def clean_employee_text(inc_employee: employee) -> tuple[employee, float]:
def clean_employee_text(inc_employee: Employee) -> Employee:

    resume_text=inc_employee.cleansed
    resume_text = re.sub('httpS+s*', ' ', resume_text)  # remove URLs
    resume_text = re.sub('RT|cc', ' ', resume_text)  # remove RT and cc
    resume_text = re.sub('#S+', '', resume_text)  # remove hashtags
    resume_text = re.sub('@S+', '  ', resume_text)  # remove mentions
    resume_text = re.sub(r'\r', '', resume_text)
    resume_text = re.sub(r'\n', '', resume_text)
    resume_text = re.sub(r'\t', ' ', resume_text) #remove tabs    
    resume_text = re.sub(' +', ' ', resume_text) # remove extra whitespace
    resume_text = re.sub(r'[<>&\'"()]', '', resume_text)
    resume_text.rstrip()
    resume_text.lstrip()
    resume_text = re.sub('[%s]' % re.escape("""!"#$%&'()*+,-./:;<=>?@[]^_`{|}~"""), ' ', resume_text)  # remove punctuations
    resume_text = ''.join([i if ord(i) < 128 else ' ' for i in resume_text])
    resume_text=re.sub(r'\W+', ' ', resume_text)
    resume_text=re.sub(' +', ' ', resume_text)
    inc_employee.cleansed=resume_text
    end_t=perf_counter

    return inc_employee

## Read the contents of a text input and remove URL's, extra spaces, carriage returns, etc..
#
#  @param (employee)            - Employee Class (to pull data)
#  @returns (employee, float)   - Altered employee record, duration of execution
#def clean_text2(inc_employee: employee) -> tuple[employee, float]:
def clean_text2(inc_employee: Employee) -> Employee:

    resume_text=inc_employee.original
    
    resume_text = re.sub('httpS+s*', ' ', resume_text)  # remove URLs
    resume_text = re.sub('RT|cc', ' ', resume_text)  # remove RT and cc
    resume_text = re.sub('#S+', '', resume_text)  # remove hashtags
    resume_text = re.sub('@S+', '  ', resume_text)  # remove mentions
    resume_text = re.sub(r'\r', '', resume_text)
    resume_text = re.sub(r'\n', '', resume_text)
    resume_text = re.sub(r'\t', ' ', resume_text) #remove tabs
    resume_text = re.sub(' +', ' ', resume_text) # remove extra whitespace
    resume_text = re.sub(r'[<>&\'"()]', '', resume_text)
    
    resume_text.rstrip()
    resume_text.lstrip()
    resume_text = re.sub('[%s]' % re.escape("""!"#$%&'()*+,-./:;<=>?@[]^_`{|}~"""), ' ', resume_text)  # remove punctuations
    resume_text = ''.join([i if ord(i) < 128 else ' ' for i in resume_text])

    resume_text=re.sub(r'\W+', ' ', resume_text)
    resume_text=re.sub(' +', ' ', resume_text)       
    # Remove punctuation
    #no_punctuation = (nopunc.translate(str.maketrans('', '', string.punctuation)) for nopunc in lower)
    #resumeText = ''.join(x for x in resumeText if x.isalnum())
    #resumeText = re.sub(r'[^x00-x7f]',r' ', resumeText) 
    #resumeText = re.sub('s+', ' ', resumeText)  # remove extra whitespace
    inc_employee.cleansed=resume_text
    end_t=perf_counter

    return inc_employee

def clean_text(inc_str: str) -> str:
    resumeText = re.sub('httpS+s*', ' ', inc_str)  # remove URLs
    resumeText = re.sub('RT|cc', ' ', resumeText)  # remove RT and cc
    resumeText = re.sub('#S+', '', resumeText)  # remove hashtags
    resumeText = re.sub('@S+', '  ', resumeText)  # remove mentions
    resumeText = re.sub(r'\r', '', resumeText)
    resumeText = re.sub(r'\n', '', resumeText)
    resumeText = re.sub(' +', ' ', resumeText) # remove extra whitespace
    resumeText = re.sub(r'[<>&\'"()]', '', resumeText)

    resumeText.rstrip()
    resumeText.lstrip()
    resumeText = re.sub('[%s]' % re.escape("""!"#$%&'()*+,-./:;<=>?@[]^_`{|}~"""), ' ', resumeText)  # remove punctuations
    resumeText = ''.join([i if ord(i) < 128 else ' ' for i in resumeText])

    resumeText=re.sub(r'\W+', ' ', resumeText)
    resumeText=re.sub(' +', ' ', resumeText)       
    # Remove punctuation
    #no_punctuation = (nopunc.translate(str.maketrans('', '', string.punctuation)) for nopunc in lower)
    #resumeText = ''.join(x for x in resumeText if x.isalnum())
    #resumeText = re.sub(r'[^x00-x7f]',r' ', resumeText) 
    #resumeText = re.sub('s+', ' ', resumeText)  # remove extra whitespace
    return resumeText

## Read the contents of a text input remove stop words (nltk)
#
#  @param (Text for text to clean) - str    - Incoming text without modification.
#  @returns (Transformed text)     - str    - Stop words removed
def clean_stop_words(inc_str:str) -> str:
        filtered_list = []
        response=word_tokenize(inc_str)
        wordlist = [x for x in response if (len(x)>=2 and x.isalpha())]
        for word in wordlist:
            if word.casefold() not in stop_words:
              filtered_list.append(word)
        return str(' '.join(filtered_list))

## Read the contents of a text input and remove extra spaces
#
#  @param (Text for text to clean) - str    - Incoming text without modification.
#  @returns (Transformed text)     - str    - Extra spaces removed
def clean_string (inc_str:str) -> str:
        response=re.sub(r'\W+', ' ', inc_str)
        response=re.sub(' +', ' ', response)           
        return str(response)

## Read the contents of a text input modify string for stem words (nltk)
#
#  @param (Text for text to clean) - str    - Incoming text without modification.
#  @returns (Transformed text)     - str    - Stem words altered
def clean_stem_words(inc_str:str) -> str:
        filtered_list = []
        response=word_tokenize(inc_str)
        wordlist = [x for x in response if (len(x)>=2 and x.isalpha())]
        stemmed_words = [stemmer.stem(word) for word in wordlist]
        return str(' '.join(stemmed_words))

## Read the contents of a text input remove stop words (nltk)
#
#  @param (Text for text to clean) - str    - Incoming text without modification.
#  @returns (Transformed text)     - str    - Stop words removed
def clean_lemmatizer_words(inc_str:str) -> str:
        filtered_list = []
        response=word_tokenize(inc_str)
        wordlist = [x for x in response if (len(x)>=2 and x.isalpha())]
        lemmatized_words = [lemmatizer.lemmatize(word) for word in wordlist]
        return str(' '.join(lemmatized_words))

## Read the contents of a text and completely break it down to the lowest level
#
#  @param (Text for text to clean) - str    - Incoming text without modification.
#  @returns (Transformed text)     - str    - Complete purge of all content for NLP
def cleanse_string(inc_str: str) -> str:
    response=clean_string(inc_str)
    response=clean_text(response)
    response=clean_stop_words(response)
    response=clean_stem_words(response)
    response=clean_lemmatizer_words(response)
    response=word_tokenize(response)
    return response

#### Spell Checker (rudimentary)

In [11]:
## Receives body of text and perform spelling analysis on results returning % misspelled.  There will never be 0.0%
#
#  @param (employee)     - employee - Class to encapsulate data
#  @returns ({})         - employee
def spelling(inc_employee: Employee) -> Employee:

    # find those words that may be misspelled
    words=word_tokenize(clean_string(inc_employee.original).lower())
    #now make sure everything is alphabetical and remove 3 letter words that might be acronyms
    wordlist = [x for x in words if (len(x)>3 and x.isalpha())]
        
    #now find the unique words to shorten the list
    wordlist=list(set(wordlist))
    wordlist.sort()
        
    misspelled_words=[]
    for idx, word in enumerate(wordlist):
        misspelled = spell.unknown([word])
        if len(misspelled) > 0:
            misspelled_words.append(word)
            #recommendation=spell.candidates(word)

    #sometimes garbage resumes (pictures) return no words and can cause division by zero
    percent_error=0.0
    try:
        percent_error=len(misspelled_words) / len(wordlist) 
    except Exception as e:
        process_exception(e)
        percent_error=-999.0
    finally:
        inc_employee.spelling=percent_error

    return inc_employee
    

#### Read PDF Resumes

In [12]:
## Read the contents of a resume in PDF format. Unstructured data.
#  https://towardsdatascience.com/extracting-text-from-pdf-files-with-python-a-comprehensive-guide-9fc4003d517
#
#  @param (Text for filename, str) - str    - Filename for target resume.
#  @returns ({})                   - dict   - Results with metrics
def read_pdf(inc_filename:str) -> {}:

    start_t = perf_counter()
    #debug.msg_info(f"Entering {__name__} {inspect.stack()[0][3]}")

    try:
        if not ( os.path.isfile(inc_filename) ):
            debug.msg_error(f"ERROR detected, the input data file for work further in the notebook is missing.  Aborting execution.")
            debug.msg_error(f"  Resolve the {inc_filename} missing file and repeat.")
            raise SystemExit("Unable to continue without data.")
    except Exception as e:
        process_exception(f"ERROR detected trying detect the PDF path as follows: {str(e)}")

    doc = fitz.open(inc_filename)
    output = []
    total_text=""

    for page in doc:
        output += page.get_text("blocks")

        previous_block_id = 0 # Set a variable to mark the block id
        total_text=""

        for block in output:
             if block[6] == 0: # We only take the text
                  #if previous_block_id != block[5]: # Compare the block number
                  #    print("\n")
                  plain_text = str(unidecode(block[4]))
                  #handle hyphenations and slashes
                  plain_text = " ".join( plain_text.split("/") )
                  plain_text = " ".join( plain_text.split("-") )
                  total_text=" ".join([total_text, plain_text])

    #debug.msg_info(f"Exiting {__name__} {inspect.stack()[0][3]}")
    end_t = perf_counter()

    resultant={}
    resultant['CANDIDATE']=Path(inc_filename).stem
    resultant['RESUME']=total_text
    resultant['DURATION']=end_t-start_t

    return resultant


#### Read PDF Position Descriptions (PDs)

In [13]:
## Given a filename read a Position Description, unstructured data and save to a data structure.
#
#  @param (Text for filename, str) - str    - Filename for target Position Description.
#  @returns ({})                   - Dictionary of results
def read_pd_pdf(inc_filename:str) -> {}:

    start_t = perf_counter()
    try:
        if not ( os.path.isfile(inc_filename) ):
            print(f"ERROR detected, the input data file for work further in the notebook is missing.  Aborting execution.")
            print(f"  Resolve the {inc_filename} missing file and repeat.")
            raise SystemExit("Unable to continue without data.")
    except Exception as e:
        process_exception(f"ERROR detected trying detect the PDF path as follows: {str(e)}")

    doc = fitz.open(inc_filename)
    output = []
    total_text=""

    for page in doc:
        output += page.get_text("blocks")
        total_text=""

        for block in output:
             if block[6] == 0: # We only take the text
                  plain_text = str(unidecode(block[4]))
                  #handle hyphenations and slashes
                  plain_text = " ".join( plain_text.split("/") )
                  plain_text = " ".join( plain_text.split("-") )
                  total_text=" ".join([total_text, plain_text])

    end_t = perf_counter()
    resultant={}
    resultant['NAME']=Path(inc_filename).stem
    resultant['PD']=total_text
    resultant['DURATION']=end_t-start_t

    return resultant


#### Prompt Injection Defense

Using pre-trained neural layers via hugging face and keyword searches perform project injection inspection.

In [14]:
## Chop up a string based on chunk size and return an array of strings
#
#  @param (Incoming String to Chop)    - str
#  @param (Chunk Size)                 - int
#  @returns ([])                       - list 
def prompt_injection_split_string(your_string, n) -> []:
    return [your_string[i:i + n] for i in range(0, len(your_string), n)]

In [15]:
## Interprets model results for project injection attacks.
#
#  @param (Transformer Pipeline)    - pipeline - Mechanism via "transformer" library to read neural layer and execute evaluation on it.
#  @param (Text to Analyze, String) - str      - Actual input to evaluate.
#  @returns ({})                    - dict     - Results of neural processing, dictionary of true/false:% quality response
def prompt_injection_predict(inc_pipeline: pipeline, inc_prompt: str) -> {}:
    id2label = {
    'LEGIT':    False,
    'POSITIVE': False,
    'LABEL_1':  False,
    'SAFE':     False,
    
    'INJECTION':True,
    'NEGATIVE': True,
    'LABEL_0':  True,
    'UNSAFE':   True,
    }
    return {id2label[x['label']]: x['score'] for x in inc_pipeline(inc_prompt)}

In [16]:
## Looks for project injection using various neural layers
#
#  @param (Text to Analyze, String) - String - Actual input to evaluate.
#  @returns (String)                - String - Transformed string abstracting name of person.
def detect_PromptInjection(inc_prompt:str) -> bool:

    resultant=False
    offending_content=[]
    keywords=["ignore", "pretend" ]
    prompt_detected=[]
    #debug.msg_info(f"Entering {__name__} {inspect.stack()[0][3]}")

    try:
        chunk_size=prompt_defense_model_chunk_size
        chunks=prompt_injection_split_string(inc_prompt, chunk_size)
        for model_idx,model_name in enumerate(the_models):

            debug.msg_debug(f"......processing model {PROMPT_INJECTION_MODELS[model_idx]}")
            for chunk_idx, chunk_value in enumerate(chunks):
                the_answer=prompt_injection_predict(the_models[model_idx],str(chunk_value))
                #if an offending answer is found store it for future analysis.
                if (True in list(the_answer.keys())):
                    offending_content.append(chunk_value)
                prompt_detected.append(the_answer)
            #print(results)

    except Exception as e:
        process_exception(f"ERROR predict prompt injection as follows: {str(e)}")
        prompt_detected.append({False:100.0})

    for status in prompt_detected:
        for the_status in status.keys():
            if (the_status):
                resultant=True

    debug.msg_debug(f"......evaluating keywords")
    wordlist=word_tokenize(inc_prompt.lower())
    for word in wordlist:
        if word in keywords:
            resultant=True

    #debug.msg_info(f"Exiting {__name__} {inspect.stack()[0][3]}")
    return resultant, offending_content


#### Privacy Information Defense

Look for Spacy entities known as "PERSON" for names of people and using regular expression library to find related PII information and abstract.

In [17]:
## Looks for PERSON object identified by spacy and abstracts the input to a constant "NAME"
#  https://www.geeksforgeeks.org/python-named-entity-recognition-ner-using-spacy/
#  @param (Spacy Model for Parsing) - Spacy Model
#  @param (Text to Analyze, String) - String - Actual input to evaluate.
#  @returns (String)                - String - Transformed string abstracting name of person.
def clean_named_entity_recognition(inc_model, inc_text) -> str:

    doc = inc_model(inc_text)
    #nec_labels=["PERSON", "ORG", "DATE", "TIME"]
    nec_labels=["PERSON"]

    text_chunks=[]
    cleansed_text=inc_text
    if len(inc_text) > inc_model.max_length:
        chunks=int(round(len(inc_text)/inc_model.max_length))
        start=0
        end=inc_model.max_length
        for idx, chunk in enumerate(chunks):
            start=idx*inc_model.max_length
            end=(idx+1) * inc_model.max_length
            text_chunks.append(inc_text[start:end])
    else:
        text_chunks.append(inc_text)

    try:
        for idx, chunk in enumerate(text_chunks):
            doc=inc_model(chunk)    
            for ent in doc.ents:
                if ent.label_ in nec_labels:
                    #debug.msg_warning(f" Encountered NER {ent}")
                    cleansed_text = re.sub(f"{ent}", f"{str(ent.label_)}", cleansed_text)
    except Exception as e:
        debug.msg_warning(f"clean_named_entity_recognition threw an exception: {repr(e)}")
        pass  #continue processing regardless, we'll accept loss of some Person identification to keep the code processing
        
    return(cleansed_text)

In [18]:
## Looks for a variety of specific criteria to transform PII
#  Could consider adding data to the domain using this technique: https://github.com/lmeulen/PrivacyFilter/tree/master
#  @param (Text to Analyze, String) - String - Actual input to evaluate.
#  @returns (String)                - String - Transformed string abstracting name of person.
def clean_pii(inc_employee:Employee) -> Employee:

    #names, countris, locations, date, time
    cleansed_text=inc_employee.cleansed

    #other PII data transformed with regular expressions
    cleansed_text=re.sub(email, "EMAIL", cleansed_text)
    cleansed_text=re.sub(credit_card, "CREDITCARD", cleansed_text)
    cleansed_text=re.sub(link, "URL", cleansed_text)
    cleansed_text=re.sub(ip, "IP", cleansed_text)
    cleansed_text=re.sub(ipv6, "IPV6", cleansed_text)
    cleansed_text=re.sub(phone, "PHONENUMBER", cleansed_text)
    cleansed_text=re.sub(street_address, "STREETADDRESS", cleansed_text)
    cleansed_text=re.sub(btc_address, "BTCADDRESS", cleansed_text)

    cleansed_text=re.sub('^\d{5}(?:[-\s]\d{4})?$',"ZIPCODE",cleansed_text)
    cleansed_text=re.sub(' \d{5}(?:[-\s]\d{4})?'," ZIPCODE2",cleansed_text)
    cleansed_text=re.sub(' .. \d{5}. ',"ZIPCODE3",cleansed_text)
   
    inc_employee.cleansed=cleansed_text
    return inc_employee

In [19]:
## Read all content from the intended comment and normalize distribution with SpaCy
#
#  @param    (inc_text)       - str          - Actual text to normalize.
#  @param    (inc_model)      - Spacy Model  - Model loaded to perform analysis of data.
#  @returns  (str)            - str          - Modified resulting text with comment field normalized.
def normalize_comments_spacy(inc_text: str, inc_model)-> str:
    try:
            new_words=[]
            text_chunks=[]
            if len(inc_text) > inc_model.max_length:
                chunks=int(round(inc_text/inc_model.max_length))
                start=0
                end=inc_model.max_length
                for idx, chunk in enumerate(chunks):
                    start=idx*inc_model.max_length
                    end=(idx+1) * inc_model.max_length
                    text_chunks.append(inc_text[start:end])
            else:
                text_chunks.append(inc_text)
            try:
                for idx, chunk in enumerate(text_chunks):
                    doc=inc_model(chunk)
                    for word in doc:
                        new_words.append(str(word))
            except Exception as e:
                debug.msg_warning(f"Normalize comments threw an exception: {repr(e)}")
                pass  #continue processing regardless, we'll accept loss of some Person identification to keep the code processing
            resultant=" ".join(new_words)
    except Exception as e:
        debug.msg_warning(f"Normalize comments threw an exception: {repr(e)}")
        process_exception(e)
        
    return resultant

In [20]:
## Looks for EMAIL object identified by spacy and abstracts the input to a constant "EMAILADDR"
#  https://spacy.pythonhumanities.com/02_02_matcher.html#:~:text=How%20to%20use%20the%20spaCy%20Matcher%201%206.1.,...%205%206.5.%20Finding%20Quotes%20and%20Speakers%20
#  @param (Spacy Model for Parsing)         - Spacy Model
#  @param (Spacy Matcher for Email Pattern) - Spacy Matcher
#  @param (Text to Analyze, String)         - String - Actual input to evaluate.
#  @returns (String)                        - String - Transformed string abstracting email
def clean_email_spacy(inc_model, inc_matcher, inc_text) -> str:

    text_chunks=[]
    cleansed_text=inc_text
    if len(inc_text) > inc_model.max_length:
        chunks=int(round(len(inc_text)/inc_model.max_length))
        start=0
        end=inc_model.max_length
        for idx, chunk in enumerate(chunks):
            start=idx*inc_model.max_length
            end=(idx+1) * inc_model.max_length
            text_chunks.append(inc_text[start:end])
    else:
        text_chunks.append(inc_text)

    try:
        for idx, chunk in enumerate(text_chunks):
            doc=inc_model(chunk)
            matches = inc_matcher(doc)            
            for match in matches:
                cleansed_text = re.sub(f"{str(doc[match[0]:match[1]])}", f"EMAIL", cleansed_text)
    except Exception as e:
        debug.msg_warning(f"clean_email_spacy threw an exception: {repr(e)}")
        pass  #allow it to continue processing, we have other was of managing email
    
    return(cleansed_text)

#### Read all PDs and store to dictionary

In [21]:
## Iterates through data directory reading PDF's in target location (must be PD's and PDF's), returns hashmap (dictionary) of PD's
#
#  @param (None)
#  @returns (dict{})  - Dictionary of resumes saved to each key, keys are the filename of the PD
def get_pds(*args) -> {}:

    pd_domains={}
    pd_filenames=[]
    the_pd_titles=['IT Specialist GS-2210-12', ]

    debug.msg_info(f"Entering {__name__} {inspect.stack()[0][3]}")

    if len(args) == 0:
        try:
            if not ( os.path.exists(DATA_DIR) ):
                os.mkdir(DATA_DIR)
        except Exception as e:
            process_exception(f"ERROR detected trying to create the root DATA path as follows: {repr(e)}")
    
        pd_pdf_path=f"{DATA_DIR}{os.sep}pd"             #PD_EXAMPLE.pdf
    
    
        if os.path.exists(pd_pdf_path):
            for file in os.listdir(pd_pdf_path):
                if str(file).endswith(".pdf") or str(file).endswith(".PDF"):
                    pd_filenames.append(os.path.join(pd_pdf_path, file))
        else:
            debug.msg_error("Directory {pd_pdf_path} not found, likely won't get any data.")
    else:
        if os.path.isfile(args[0]):
            pd_filenames.append(args[0])
        else:
            debug.msg_error("File {args[0]} not found, likely won't get any data.")
        

    debug.msg_debug(f"{len(pd_filenames)} PD files read in.")

    #read resumes, don't alter content
    debug.msg_debug("Reading resumes...")
    with Pool() as pool:
        results = pool.map(read_pd_pdf, pd_filenames)
        for idx, dictionary in enumerate(results):
            pd_name=dictionary['NAME']
            pd_domains[pd_name]=dictionary['PD']
            debug.msg_debug(f"...read {pd_name} in {dictionary['DURATION']:.2} s.")

    #read PD's, don't alter text
    for idx,filename in enumerate(pd_filenames):
        debug.msg_debug(f"...processing {filename}")

    debug.msg_info(f"Exited {__name__} {inspect.stack()[0][3]}")

    return pd_domains


#### Read all Resumes and store to Employee Class (list)

In [22]:
## Iterates through data directory reading PDF's in target location (must be resume's only and PDF's), returns list of Employee Class
#
#  @param (None)
#  @returns (list[])  - List of Employee Class with resume data name, original populated

def get_resumes(*args) -> []:

    employees=[]
    resume_filenames=[]
    resume_pdf_path=f"{DATA_DIR}{os.sep}resume"

    debug.msg_info(f"Entering {__name__} {inspect.stack()[0][3]}")
    if len(args) == 0:
        try:
            if not ( os.path.exists(DATA_DIR) ):
                os.mkdir(DATA_DIR)
        except Exception as e:
            process_exception(f"ERROR detected trying to create the root DATA path as follows: {repr(e)}")
    
        if os.path.exists(resume_pdf_path):
            for file in os.listdir(resume_pdf_path):
                if file.endswith(".pdf") or file.endswith(".PDF"):
                    resume_filenames.append(os.path.join(resume_pdf_path, file))
        else:
            debug.msg_error(f"Directory {resume_pdf_path} not found, likely won't get any data.")
    else:
        if os.path.isfile(args[0]):
            resume_filenames.append(args[0])
        else:
            debug.msg_error("File {args[0]} not found, likely won't get any data.")
        
    debug.msg_debug(f"{len(resume_filenames)} Resume files read in.")

    ####################################################################################################################
    #- READ resumes from PDF, store to employee class
    #read resumes, don't alter content
    ####################################################################################################################
    debug.msg_debug("Reading resumes...")
    with Pool() as pool:
        results = pool.map(read_pdf, resume_filenames)
        for idx, dictionary in enumerate(results):
            debug.msg_debug(f"...read {dictionary['CANDIDATE']} completed in {dictionary['DURATION']:.2f}s")
            #local_emp=Employee(name=dictionary['CANDIDATE'],original=dictionary['RESUME'], cleansed="")
            local_emp=Employee(dictionary['CANDIDATE'],dictionary['RESUME'], "")
            #clean up carriage returns from the output so the final datafile is "clean" and doesn't cause prompt issues.
            local_emp.original = re.sub("\n", "  ", local_emp.original)
            local_emp.original = local_emp.original.strip()
            employees.append(local_emp)

    ####################################################################################################################
    #- Spell Check
    ####################################################################################################################
    debug.msg_debug("Peforming Spellcheck...")
    spell_employees=[]
    with Pool() as pool:
        results = pool.map(spelling, employees)
        employees=results
 
    ####################################################################################################################
    #- Setup SpaCy model (load)
    #clean PII before regular cleansing as PII requires semantic context.
    #had to run spacy as a singular process, would NOT work multi-process
    ####################################################################################################################
    try:
        spacy.prefer_gpu()
    except Exception as e:
        #no gpu available will default to CPU
        #CGW, FIX, TODO - could add number of CPU's to allow for maximum processing, consider this.
        pass
    #nlp =spacy.load(SPACY_MODEL)
    #nlp =spacy.load(SPACY_MODEL, disable=["tok2vec", "tagger", "parser", "attribute_ruler"])
    nlp =spacy.load(SPACY_MODEL, disable=['tagger', 'lemmatizer', 'textcat'])

    ####################################################################################################################
    #- REMOVE PERSON DATA
    ####################################################################################################################
    debug.msg_info("Remove person from text.")
    for emp_idx, emp_instance in enumerate(employees):
        cleansed_output=clean_named_entity_recognition(nlp,emp_instance.original)
        employees[emp_idx].cleansed=cleansed_output

    ####################################################################################################################
    #- REMOVE EMAIL
    ####################################################################################################################
    debug.msg_info("Removing email from text.")
    matcher = Matcher(nlp.vocab)
    pattern = [{"LIKE_EMAIL": True}]
    matcher.add("EMAIL_ADDRESS", [pattern])

    ####################################################################################################################
    #- Normalize Words
    ####################################################################################################################
    debug.msg_info("Use SpaCy to normalize comments.")    
    for emp_idx, emp_instance in enumerate(employees):
        cleansed_output=normalize_comments_spacy(emp_instance.cleansed, nlp)
        employees[emp_idx].cleansed=cleansed_output

    ####################################################################################################################
    #- Check All PII
    ####################################################################################################################
    #clean PII before regular cleansing as PII requires semantic context.
    debug.msg_debug("Cleaning remaining PII Content..2nd Pass.")
    cleaned_employees=[]
    with Pool() as pool:
        results = pool.map(clean_pii, employees)
        employees=results

    #print("##################################################")
    #print(employees[0].cleansed)
    #print("##################################################")

    ####################################################################################################################
    #- Clean actual text of various characters.
    ####################################################################################################################
    debug.msg_debug("Cleaning Resume Content...")
    with Pool() as pool:
        results = pool.map(clean_employee_text, employees)
        employees=results

    #print("##################################################")
    #print(employees[0].cleansed)
    #print("##################################################")

    ####################################################################################################################
    #- Prompt Injection Defense
    ####################################################################################################################
    for the_idx, the_employee in enumerate(employees):
        prompt_found, prompt_content=detect_PromptInjection(the_employee.cleansed)
        debug.msg_debug(f"...processing {the_employee.name} data for prompt injection.")
        if (prompt_found):
            debug.msg_warning(f"......{BOLD_START}(CAUTION):{BOLD_END} - {the_employee.name} might contain content that attempts a prompt injection attack.")
            debug.msg_warning(f".........potential attack: {prompt_content}")
            the_employee.project_injection_detected=True
            the_employee.prompt_injection=prompt_content
    
    debug.msg_info(f"Exiting {__name__} {inspect.stack()[0][3]}")

    return employees



In [23]:
#@st.cache_resource()
def load_model(inc_model_name):
    model = pipeline("text-classification", model=str(model_name), device=device)
    return model

In [ ]:
def set_global_configuration() -> None:

    debug.msg_info(f"Entered {__name__} {inspect.stack()[0][3]}")
    ############################################
    # GLOBAL CONFIGURATION
    ############################################
    #used for values outside standard ASCII, just do it, you'll need it                       
    os.environ['PYTHONIOENCODING']=str(os.getenv("FORMAT_ENCODING"))
    #spacy requirement
    os.environ['TOKENIZERS_PARALLELISM']=str(os.getenv("SPACY_TOKENIZERS_PARALLELISM"))

    debug.msg_info("Variable declaration.")    
    ############################################
    # GLOBAL VARIABLES
    ############################################
    global DEBUG, DEBUG_DATA
    DEBUG=bool(os.getenv("DEBUG"))
    DEBUG_DATA=bool(os.getenv("DEBUG_DATA"))
    
    # CODE CONSTRAINTS
    global VERSION_NAME, VERSION_MAJOR, VERSION_MINOR, VERSION_RELEASE
    VERSION_NAME    = str(os.getenv("VERSION_NAME"))
    VERSION_MAJOR   = str(os.getenv("VERSION_MAJOR"))
    VERSION_MINOR   = str(os.getenv("VERSION_MINOR"))
    VERSION_RELEASE = str(os.getenv("VESION_RELEASE"))
    
    #used for values outside standard ASCII, just do it, you'll need it
    global TEXT_WIDTH, BOLD_START, BOLD_END
    TEXT_WIDTH      =str(os.getenv("FORMAT_TEXT_WIDTH"))
    BOLD_START      =str(os.getenv("FORMAT_BOLD_START"))
    BOLD_END        =str(os.getenv("FORMAT_BOLD_END"))
    
    ###########################################
    #- API Parameters for things like WordCloud
    ###########################################
    global IMG_BACKGROUND, IMG_FONT_SIZE_MIN, IMG_WIDTH, IMG_HEIGHT
    IMG_BACKGROUND   =str(os.getenv("IMG_BACKGROUND"))
    IMG_FONT_SIZE_MIN=int(os.getenv("IMG_FONT_SIZE_MIN"))
    IMG_WIDTH        =int(os.getenv("IMG_WIDTH"))
    IMG_HEIGHT       =int(os.getenv("IMG_WIDTH"))
    
    ############################################
    # APPLICATION VARIABLES
    ############################################                                  
    global PROJECT_ID, BUCKET_ID, LOCATION, SPELL_CHECK_DISTANCE, MINIMUM_AI_WAIT, DATA_DIR, OUTPUT_DIR, EXCEL_CHAR_BOUNDARY, DELIM
    PROJECT_ID          = str(os.getenv("CLD_PROJECT_ID"))
    BUCKET_ID           = str(os.getenv("CLD_BUCKET_ID"))
    LOCATION            = str(os.getenv("CLD_LOCATION"))
    SPELL_CHECK_DISTANCE= int(os.getenv("SPELL_CHECK_DISTANCE"))
    MINIMUM_AI_WAIT     = int(os.getenv("TIME_MINIMUM_AI_WAIT"))
    DATA_DIR            = str(os.getenv("DATA_INPUT_DIR"))
    OUTPUT_DIR          = str(os.getenv("DATA_OUTPUT_DIR"))
    EXCEL_CHAR_BOUNDARY = str(os.getenv("EXCEL_CHAR_BOUNDARY"))
    DELIM               = str(os.getenv("IO_DELIM"))
    
    print(PROJECT_ID)
    print(BUCKET_ID)
    print(LOCATION)
    
    """
    ############################################
    # GENERATIVE MODEL PARAMETERS
    ############################################
    the_model="unknown"
    model_temperature=0.0
    model_max_tokens=0
    model_max_token_response=0
    model_top_p=0.0
    model_frequency_penalty=0
    model_presence_penalty=0
    summary_token_max=0
    if environments[user_input]=="GCP":
        #model parameters
            #Gemini 1.5 Flash	google/gemini-1.5-flash-001 
            #Gemini 1.5 Prov	google/gemini-1.5-pro-001
            #Gemini 1.0 Prov	google/gemini-1.0-pro-002
            #                   google/gemini-1.0-pro-001
            #                   google/gemini-1.0-pro
        #the_model="gemini-1.5-pro-001"
        the_model="gemini-1.5-flash"
        model_temperature=1.0
        model_max_tokens=8000
        model_max_token_response=8000
        model_top_p=0.95
        model_frequency_penalty=0
        model_presence_penalty=0
        summary_token_max=150
    else:
        #model parameters
        the_model="gpt-35-turbo-16k"
        model_temperature=1.0
        model_max_tokens=8000
        model_max_token_response=2000
        model_top_p=0.95
        model_frequency_penalty=0
        model_presence_penalty=0
        summary_token_max=150
    ############################################
    # PROMPT PARAMETERS
    ############################################
    PROMPT_SUMMARY_LIMIT="1000"                   #number of words to generate
    PROMPT_SUMMARY_METHOD=" abstractive "        #abstractive or extractive
    PROMPT_INJECTION_MODELS=[
                             #https://python.langchain.com/v0.1/docs/guides/productionization/safety/hugging_face_prompt_injection/
                             #from optimum.onnxruntime import ORTModelForSequenceClassification
                             "protectai/deberta-v3-base-prompt-injection-v2",
                            ]    
    """
    
    debug.msg_info(f"Exited {__name__} {inspect.stack()[0][3]}")


In [24]:
## Main routine that executes all code, does return a data frame of data for further analysis if desired.
#
#  @param (None)
def process() -> None:

    debug.msg_info(f"Entering {__name__} {inspect.stack()[0][3]}")

    employees=[]
    pds={}

    #read in all the resumes, clean the pii and text once for future save
    #employees=get_resumes("./ResumeZahidChaudhry.pdf")
    employees=get_resumes()

    #we trust the PDs because we control and own that data, no alteration of the data
    #pds=get_pds("./ZahidTargetPD.pdf")
    pds=get_pds()

    #establish data version, aligned with code
    data_version_release="-".join([str(VERSION_NAME), str(VERSION_MAJOR), str(VERSION_MINOR), str(VERSION_RELEASE)])

    #save results of all data to binary file
    debug.msg_debug("Saving cleansed/prepped data to binary file.")
    target_filename=f"./{data_version_release}"+"_employee.bin"
    try:
        pickle.dump(employees, open(target_filename, "wb"))
        me = pickle.load(open(target_filename, "rb"))
    except (pickle.UnpicklingError, FileNotFoundError, IOError, Exception)  as e:
        debug.msg_warning("FAILED to unpickle the saved binary file, you might have corruption, investigate.")
        process_exception(e)
    debug.msg_debug(f"...saved and reloaded {target_filename}")

    #position descriptions
    target_filename=f"./{data_version_release}"+"_pds.bin"
    try:
        pickle.dump(pds, open(target_filename, "wb"))
        me = pickle.load(open(target_filename, "rb"))
    except (pickle.UnpicklingError, FileNotFoundError, IOError, Exception)  as e:
        debug.msg_warning("FAILED to unpickle the saved binary file, you might have corruption, investigate.")
        process_exception(e)
    debug.msg_debug(f"...saved and reloaded {target_filename}")

    debug.msg_info(f"Exited {__name__} {inspect.stack()[0][3]}")


#### Main Routine (call all other routines)

### Pythonic way of calling main if we go to a script

In [25]:
if __name__ == "__main__":

    set_library_configuration()
    start_t=perf_counter()
    print("BEGIN PROGRAM")
    
    ############################################
    # GPU Setup (for multiple GPU devices)
    ############################################
    #THE_DEVICE=0
    #os.environ['CUDA_VISIBLE_DEVICES']=f"${THE_DEVICE}"
    #device = torch.device(f"cuda:{THE_DEVICE}") 
    device = torch.cuda.current_device()
    torch.cuda.empty_cache()

    ############################################
    # SECRETS & ENV VARIABLES
    ############################################
    set_library_configuration()
    start_t=perf_counter()
    print("BEGIN PROGRAM")

    ############################################
    # SECRETS & ENV VARIABLES
    ############################################
    load_dotenv("../.env")
    load_dotenv("../.env_ai")
    load_dotenv("../.env_cloud")
    load_dotenv("../.env_api_keys")
    load_dotenv("./.env_app")    

    set_global_configuration()
    
    print(f"PROJECT_ID")
    print(f"BUCKET_ID")
    print(f"LOCATION")
    
    #with warnings.catch_warnings():
    # To ignore specific warning types:
    warnings.filterwarnings('ignore', category=DeprecationWarning)
    warnings.filterwarnings('ignore', category=FutureWarning)
    warnings.filterwarnings('ignore', category=UserWarning)
    

    ############################################
    # PROMPT PARAMETERS
    ############################################
    PROMPT_SUMMARY_LIMIT="200"                   #number of words to generate
    PROMPT_SUMMARY_METHOD=" abstractive "        #abstractive or extractive
    JSON_PAYLOAD = '{ "Score": 85, "Strengths": "This candidates resume is well formed.", "Weaknesses": "This candidates resume is not well rounded" }'
    PROMPT_INJECTION_MODELS=[
                             #https://python.langchain.com/v0.1/docs/guides/productionization/safety/hugging_face_prompt_injection/
                             #from optimum.onnxruntime import ORTModelForSequenceClassification
                             "protectai/deberta-v3-base-prompt-injection-v2",
                            ]
    
    ########################################
    #Define Potential "answers" from the various neural layers
    #Is the Prompt Detected?
    ########################################
    id2label = {
        'LEGIT':    False,
        'POSITIVE': False,
        'LABEL_1':  False,
        'SAFE':     False,
        
        'INJECTION':True,
        'NEGATIVE': True,
        'LABEL_0':  True,
        'UNSAFE':   True,
    }
    prompt_defense_model_chunk_size=512
    
    #python -m spacy download en
    #python -m spacy download en_core_web_sm
    #SPACY_MODEL="en_core_web_sm"
    #python -m spacy download en_core_web_lg
    #SPACY_MODEL="en_core_web_lg"
    SPACY_MODEL="en_core_web_trf"    
    
    ####################################################################################################################
    #- Setup SpaCy model (load)
    ####################################################################################################################
    try:
        spacy.prefer_gpu()
    except Exception as e:
        #no gpu available will default to CPU
        #CGW, FIX, TODO - could add number of CPU's to allow for maximum processing, consider this.
        debug.msg_warning(f"GPU not utilized by SpaCy, see exception: {e}")
        pass
    
    try:
        nlp =spacy.load(SPACY_MODEL)
    except Exception as e:
        debug.msg_warning("Unable to load your model, performing a download instead.")
        #!python -m spacy download {SPACY_MODEL}
        subprocess.run(["python", "-m" , "spacy", "download", SPACY_MODEL])
        pass  #we want to download not cause a problem.
    finally:
        nlp =spacy.load(SPACY_MODEL)
    
    ############################################
    # NLTK instantiation
    ############################################
    debug.msg_debug(f"...StopWords instantiated.")
    stop_words = set(stopwords.words("english"))
    debug.msg_debug(f"...PortStemmer instantiated.")
    stemmer = PorterStemmer()
    debug.msg_debug(f"...Lemmatizer instantiated.")
    lemmatizer = WordNetLemmatizer()
    debug.msg_debug(f"...SpellChecker instantiated.")
    spell = SpellChecker(distance=SPELL_CHECK_DISTANCE)

    #to add words to the potential domain of words
    spell_it_acronyms=['3d', '9001:2008', 'ad-hoc', 'ai', 'ai/ml', 'alm', 'artifactory', 'atlassian', 'aws', 'break-down', 'burn-down', 'calibration/algorithm', 'cio', 'cm', 'cmm', 'cmm/cmmi', 'cmmi', 'cnn', 'convolutional', 'csm', 'csps', 'css', 'cyber', 'cyber-security', 'cybersecurity', 'cyverse', 'cyverse.org', 'datasets', 'deliverables', 'devnet', 'devops', 'devsecops', 'di2e', 'disa', 'dod', 'dodaf', 'ea', 'eacoe', 'end-users', 'engineer/technical', 'eos', 'esapi', 'facp', 'facp/pm', 'fine-tuning', 'gcp', 'gemini/bard', 'geospatial', 'git', 'github', 'globus', 'globus.org', 'google', 'hazmat', 'hpc', 'hyper-parameters', 'hyperspectral', 'ia', 'iavs', 'idiq', 'in-situ', 'iso', 'jenkins', 'jira', 'kick-off', 'lan', 'life-cycle', 'lifecycle', 'linux', 'llc', 'llms', 'management/continuous', 'ml', 'model/view/controller', 'multi-threaded', 'mvc', 'netcdf', 'on-site', 'owasp', 'physical/bio-optical', 'pki', 'pmp', 'pmps', 'psp', 'real-time', 'regional/basin-scale', 'rpa', 'run-time', 'sdlc', 'sei', 'sensor-reading', 'situ', 'sk-learn', 'soacp', 'solutioning', 'ssl/tls', 'stig', 'stigs', 'super-computing', 'surface/subsurface', 'tensorflow/keras', 'tsp', 'vm', 'voip', 'wbs', 'web-based', 'workflow']
    spell_it_concepts=['ai', 'alm', 'aws', 'azure', 'cisa', 'cm', 'cnn', 'convolutional', 'datasets', 'devops', 'devsecops', 'disa', 'fine-tuning', 'fortran', 'gcp', 'gemini/bard', 'gis', 'github', 'google', 'hpc', 'iavs', 'lifecycle', 'llm', 'llms', 'microsoft', 'middleware', 'ml', 'netcdf', 'rpa', 'sdlc', 'soa', 'stigs', 'super-computing', 'virtualbox', 'virtualization', 'vmware']
    spell_it_languages=['ada', 'ajax', 'ansible', 'arcgis', 'artifactory', 'asp.net', 'awk', 'c++', 'csh', 'css', 'dbase', 'debian', 'etc', 'f-77', 'f-90', 'f77', 'f77/90', 'f90', 'fortran', 'geoserver', 'geospatial', 'gis', 'graphviz', 'html', 'idl', 'iis', 'intellij', 'javafx', 'javascript', 'jboss', 'jenkins', 'jinja2', 'jsp', 'keras', 'kml', 'ksh', 'kubernetes', 'lan', 'ldap', 'linux', 'matlab', 'mongodb', 'mvc', 'mvc4', 'mysql', 'netcdf', 'nginx', 'openai', 'openam', 'openldap', 'perl', 'php', 'pki', 'postgis', 'postgresql', 'powershell', 'pv-wave', 'rdbms', 'redhat', 'redmine', 'rhel', 'rpa', 'rpc', 'sk-learn', 'solaris', 'sql', 'ssl', 'svg', 'tensorflow', 'tls', 'trac', 'ubuntu', 'uml', 'unix', 'voip', 'web-based', 'weblogic', 'websecurify', 'wpa', 'xhtml', 'xml', 'yaml', 'zope']
    spell_months=['jan', 'feb', 'mar', 'apr', 'may', 'jun', 'july', 'aug', 'sept', 'oct', 'nov', 'dev']
    spell_common_acronym=['gov', 'com']

    spell.word_frequency.load_words(spell_it_acronyms)
    spell.word_frequency.load_words(spell_it_concepts)
    spell.word_frequency.load_words(spell_it_languages)
    spell.word_frequency.load_words(spell_months)
    spell.word_frequency.load_words(spell_common_acronym)

    #setup the text wrapper
    debug.msg_debug(f"...Text Wrapper instantiated.")
    wrapper = textwrap.TextWrapper(width=TEXT_WIDTH)

    #show your libraries
    lib_diagnostics()

    ############################################
    #load the injection models for future use
    ############################################
    the_models=[]
    try:
        debug.msg_info(f"Loading prompt injection model(s)")
        for model_idx,model_name in enumerate(PROMPT_INJECTION_MODELS):
            #the_models.append(pipeline("text-classification", model=str(model_name), device=0))
            the_models.append(load_model(model_name))
            debug.msg_debug(f"...loaded {model_name}")
    except Exception as e:
        process_exception(f"ERROR detected trying to use the transformer.pipeline API to load hugging face models as follows: {str(e)}")

    ############################################
    # - Core workhorse routine
    ############################################
    process()

    end_t=perf_counter()
    print("END PROGRAM")
    print(f"Elapsed time: {end_t - start_t}")


[2024-12-05 02:23:35 UTC]    INFO: Setting Pandas and Numpy library options. 
BEGIN PROGRAM
[2024-12-05 02:23:35 UTC]    INFO: Variable declaration. 


/opt/conda/lib/python3.10/site-packages/thinc/shims/pytorch.py:253: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(filelike, map_location=dev

[2024-12-05 02:23:42 UTC]   DEBUG: ...StopWords instantiated. 
[2024-12-05 02:23:42 UTC]   DEBUG: ...PortStemmer instantiated. 
[2024-12-05 02:23:42 UTC]   DEBUG: ...Lemmatizer instantiated. 
[2024-12-05 02:23:42 UTC]   DEBUG: ...SpellChecker instantiated. 
[2024-12-05 02:23:43 UTC]   DEBUG: ...Text Wrapper instantiated. 


/var/tmp/ipykernel_627525/3795858203.py:6: DeprecationWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html
  import pkg_resources


[2024-12-05 02:23:43 UTC]    INFO: Entering __main__ lib_diagnostics 
jupyter-core                            #: 5.7.2               
langchain                               #: 0.3.1               
langchain-core                          #: 0.3.6               
nltk                                    #: 3.9.1               
numba                                   #: 0.60.0              
numpy                                   #: 1.26.4              
pandas                                  #: 2.2.3               
pydantic                                #: 2.9.2               
pyspellchecker                          #: 0.8.1               
scikit-learn                            #: 1.5.2               
scipy                                   #: 1.13.1              
seaborn                                 #: 0.13.2              
spacy                                   #: 3.7.6               
usaddress                               #: 0.5.10              
TensorFlow version                

...

[2024-12-05 02:23:45 UTC]   DEBUG: ...loaded protectai/deberta-v3-base-prompt-injection-v2 
[2024-12-05 02:23:45 UTC]    INFO: Entering __main__ process 
[2024-12-05 02:23:45 UTC]    INFO: Entering __main__ get_resumes 
[2024-12-05 02:23:45 UTC]   DEBUG: 63 Resume files read in. 
[2024-12-05 02:23:45 UTC]   DEBUG: Reading resumes... 
[2024-12-05 02:23:46 UTC]   DEBUG: ...read David Vrooman completed in 0.10s 
[2024-12-05 02:23:46 UTC]   DEBUG: ...read Clayton Allensworth completed in 0.10s 
[2024-12-05 02:23:46 UTC]   DEBUG: ...read Christopher Lombardo completed in 0.11s 
[2024-12-05 02:23:46 UTC]   DEBUG: ...read Alston Quimby completed in 0.10s 
[2024-12-05 02:23:46 UTC]   DEBUG: ...read Susan Miller completed in 0.23s 
[2024-12-05 02:23:46 UTC]   DEBUG: ...read Cheryl Hynes completed in 0.11s 
[2024-12-05 02:23:46 UTC]   DEBUG: ...read Ian McDougal completed in 0.09s 
[2024-12-05 02:23:46 UTC]   DEBUG: ...read Pasquale Lorfino completed in 0.25s 
[2024-12-05 02:23:46 UTC]   DEBUG: 

/opt/conda/lib/python3.10/site-packages/thinc/shims/pytorch.py:253: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(filelike, map_location=dev

[2024-12-05 02:23:50 UTC]    INFO: Remove person from text. 


/opt/conda/lib/python3.10/site-packages/thinc/shims/pytorch.py:114: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(self._mixed_precision):


KeyboardInterrupt: 